## Orquestación del Pipeline

El orquestador del pipeline se implementa mediante un notebook maestro que coordina la ejecución secuencial de las distintas capas del proceso ETL. Este notebook invoca cada uno de los notebooks individuales (preparación, ingestión, transformación y carga) utilizando `dbutils.notebook.run`, asegurando que cada etapa se ejecute en el orden correcto.

El flujo comienza con la preparación del entorno, donde se validan y configuran las estructuras necesarias. Posteriormente, se ejecutan los procesos de ingestión hacia la capa **bronze**, seguidos por las transformaciones hacia la capa **silver**, y finalmente la carga de datos consolidados en la capa **golden**.

El orquestador incluye control básico de errores mediante bloques `try/except`, lo que permite identificar fallos en cualquier etapa del pipeline y evitar ejecuciones inconsistentes. De esta forma, se garantiza la integridad del proceso y la trazabilidad de la ejecución.

Debido a las limitaciones de Databricks Free Edition, no se cuenta con un sistema de orquestación automatizado con scheduling o dependencias avanzadas. Sin embargo, este enfoque reproduce el comportamiento de un flujo orquestado, y en un entorno productivo podría ser sustituido por herramientas como **Databricks Workflows** o **Azure Data Factory**, incorporando monitoreo, reintentos y ejecución programada.

In [0]:
try:
    dbutils.notebook.run("/Users/roager.mx@gmail.com/ecobiciweather-databricks-etl/proceso/raw/1_Preparacion_Ambiente_Ecobici_Raw", 0)
    dbutils.notebook.run("/Users/roager.mx@gmail.com/ecobiciweather-databricks-etl/proceso/bronze/2_Ingest_Ecobici_Bronze", 0)
    dbutils.notebook.run("/Users/roager.mx@gmail.com/ecobiciweather-databricks-etl/proceso/bronze/2_Ingest_Weather_Bronze", 0)
    dbutils.notebook.run("/Users/roager.mx@gmail.com/ecobiciweather-databricks-etl/proceso/silver/3_Transform_Silver", 0)
    dbutils.notebook.run("/Users/roager.mx@gmail.com/ecobiciweather-databricks-etl/proceso/gold/4_Load_Gold", 0)
    
    print("Pipeline ejecutado correctamente")

except Exception as e:
    print("Error en pipeline:", str(e))
    raise

In [0]:
# =====================================
# VALIDADOR GENERAL DE TABLAS
# =====================================

def validate_table(table_name):
    try:
        df = spark.table(table_name)
        count = df.count()
        
        print(f"Validando {table_name} → registros: {count}")
        
        if count == 0:
            raise Exception(f"❌ Tabla {table_name} está vacía")
        
        print(f"✔ {table_name} OK")
    
    except Exception as e:
        print(f"❌ Error en {table_name}: {str(e)}")
        raise
validate_table("catalog_ecobici.bronze.ecobici")
validate_table("catalog_ecobici.silver.ecobici")
validate_table("catalog_ecobici.golden.ecobici_vs_lluvia")